In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

hf_token = "lorem-ipsum"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", token=hf_token)

In [2]:
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=quant_config,
    dtype=torch.float16,
    token=hf_token
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Task 1
GPU memory: 5202 MB

In [3]:
question = "How many planets are there in our solar system?"

tokenized_question = tokenizer.encode(question, return_tensors="pt").to("cuda")
print("Tokenized input: ", tokenized_question)

answer = model.generate(
    tokenized_question,
    return_dict_in_generate=True,
    output_scores=True
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Tokenized input:  tensor([[    1,  1602,  1287, 28312,   460,   736,   297,   813, 13024,  1587,
         28804]], device='cuda:0')


/home/student/193229/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=31) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [4]:
print("Answer tensor is: ", answer)
print("\nDecoded answer is: ")
for t in answer.sequences:
    print(tokenizer.decode(t))

Answer tensor is:  GenerateDecoderOnlyOutput(sequences=tensor([[    1,  1602,  1287, 28312,   460,   736,   297,   813, 13024,  1587,
         28804,    13,    13,  5816,   460,  5435, 28312,   297,   813, 13024,
          1587, 28723,    13,    13,  3195,   460,   272,  2955,   302,   272,
         28312]], device='cuda:0'), scores=(tensor([[-9.3516, -9.8281,  3.5039,  ..., -6.8711, -5.8203, -5.5195]],
       device='cuda:0'), tensor([[-6.0195, -5.8398,  1.7461,  ..., -1.8740, -2.7734, -2.4355]],
       device='cuda:0'), tensor([[-9.0312, -9.5938, -0.3037,  ..., -4.4531, -3.8477, -3.1875]],
       device='cuda:0'), tensor([[-8.6250, -8.6953,  2.9707,  ..., -8.5078, -7.0273, -6.9023]],
       device='cuda:0'), tensor([[-7.9766, -7.6953,  3.1660,  ..., -7.8516, -7.1211, -8.2656]],
       device='cuda:0'), tensor([[-8.4453, -8.6562,  3.4746,  ..., -7.3750, -6.0703, -7.7305]],
       device='cuda:0'), tensor([[-8.2969, -8.6797,  4.1016,  ..., -8.7500, -6.8867, -6.5000]],
       device='cu

## Task 2

In [5]:
for a in answer:
    print(a, answer[a])

sequences tensor([[    1,  1602,  1287, 28312,   460,   736,   297,   813, 13024,  1587,
         28804,    13,    13,  5816,   460,  5435, 28312,   297,   813, 13024,
          1587, 28723,    13,    13,  3195,   460,   272,  2955,   302,   272,
         28312]], device='cuda:0')
scores (tensor([[-9.3516, -9.8281,  3.5039,  ..., -6.8711, -5.8203, -5.5195]],
       device='cuda:0'), tensor([[-6.0195, -5.8398,  1.7461,  ..., -1.8740, -2.7734, -2.4355]],
       device='cuda:0'), tensor([[-9.0312, -9.5938, -0.3037,  ..., -4.4531, -3.8477, -3.1875]],
       device='cuda:0'), tensor([[-8.6250, -8.6953,  2.9707,  ..., -8.5078, -7.0273, -6.9023]],
       device='cuda:0'), tensor([[-7.9766, -7.6953,  3.1660,  ..., -7.8516, -7.1211, -8.2656]],
       device='cuda:0'), tensor([[-8.4453, -8.6562,  3.4746,  ..., -7.3750, -6.0703, -7.7305]],
       device='cuda:0'), tensor([[-8.2969, -8.6797,  4.1016,  ..., -8.7500, -6.8867, -6.5000]],
       device='cuda:0'), tensor([[-8.4609, -8.0469,  4.0742,  .

## Task 3

In [6]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

peft_tokenizer = AutoTokenizer.from_pretrained("alignment-handbook/zephyr-7b-sft-qlora", token=hf_token)
peft_model = PeftModel.from_pretrained(
    model, 
    "alignment-handbook/zephyr-7b-sft-qlora",
    token=hf_token
).to("cuda")

GPU memory: 5434 MB

## Task 4

In [7]:
print("Base PEFT template: ", peft_tokenizer.chat_template)

messages = [
    {
        "role": "user", 
        "content": "How many planets are there in our solar system?"
    },
    {
        "role": "assistant",
        "content": "There are 8 planets in Earth's solar system"
    }
]

# messages = [
#   {"role": "user", "content": "Hello, how are you?"},
#   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
#   {"role": "user", "content": "I'd like to show off how chat templating works!"},
# ]

template = peft_tokenizer.apply_chat_template(messages, tokenize=False)
print("\nTemplate:\n", template)

Base PEFT template:  {% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}

Template:
 <|user|>
How many planets are there in our solar system?</s>
<|assistant|>
There are 8 planets in Earth's solar system</s>



## Task 5

In [35]:
def get_system_prompt():
    return {
            "role": "system",
            "content": "You are helpful assistant"
        }

def get_user_message(input_str):
    return {
            "role": "user",
            "content": input_str
        }

def get_bot_message(input_str):
        return {
            "role": "assistant",
            "content": input_str
        }
    

In [ ]:
stop = False

messages = [get_system_prompt()]

while stop is not True:
    inp = input("Q: ").strip()
    if (inp == "stop"):
        stop = True
        break

    messages.append(get_user_message(inp))

    print("\n\n", messages, "\n\n")
    
    input_tensor = peft_tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    ans = peft_model.generate(
        inputs=input_tensor["input_ids"],
        max_new_tokens=128)

    full_text = peft_tokenizer.decode(ans, skip_special_tokens=True)
    text = full_text[0]
    print("\n\n", full_text, "\n\n")

    idx = text.rfind("<|assistant|>")
    resp = text[idx:].replace("<|assistant|>\n","")

    messages.append(get_bot_message(resp))
    print("A: ", resp)


Q:  what is the capital of poland


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




 [{'role': 'system', 'content': 'You are helpful assistant'}, {'role': 'user', 'content': 'what is the capital of poland'}] 




 ['<|system|>\nYou are helpful assistant \n<|user|>\nwhat is the capital of poland \n<|assistant|>\nThe capital of Poland is Warsaw.'] 


A:  The capital of Poland is Warsaw.


Q:  what is the population of poland


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




 [{'role': 'system', 'content': 'You are helpful assistant'}, {'role': 'user', 'content': 'what is the capital of poland'}, {'role': 'assistant', 'content': 'The capital of Poland is Warsaw.'}, {'role': 'user', 'content': 'what is the population of poland'}, {'role': 'assistant', 'content': 'The population of Poland is approximately 38.5 million people.'}, {'role': 'user', 'content': 'what is the population of warsaw'}] 




 ['<|system|>\nYou are helpful assistant \n<|user|>\nwhat is the capital of poland \n<|assistant|>\nThe capital of Poland is Warsaw. \n<|user|>\nwhat is the population of poland \n<|assistant|>\nThe population of Poland is approximately 38.5 million people. \n<|user|>\nwhat is the population of warsaw \n<|assistant|>\nThe population of Warsaw, the capital of Poland, is approximately 1.7 million people.'] 


A:  The population of Warsaw, the capital of Poland, is approximately 1.7 million people.


Q:  what was my first question


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




 [{'role': 'system', 'content': 'You are helpful assistant'}, {'role': 'user', 'content': 'what is the capital of poland'}, {'role': 'assistant', 'content': 'The capital of Poland is Warsaw.'}, {'role': 'user', 'content': 'what is the population of poland'}, {'role': 'assistant', 'content': 'The population of Poland is approximately 38.5 million people.'}, {'role': 'user', 'content': 'what is the population of warsaw'}, {'role': 'assistant', 'content': 'The population of Warsaw, the capital of Poland, is approximately 1.7 million people.'}, {'role': 'user', 'content': 'what was my first question'}] 




 ['<|system|>\nYou are helpful assistant \n<|user|>\nwhat is the capital of poland \n<|assistant|>\nThe capital of Poland is Warsaw. \n<|user|>\nwhat is the population of poland \n<|assistant|>\nThe population of Poland is approximately 38.5 million people. \n<|user|>\nwhat is the population of warsaw \n<|assistant|>\nThe population of Warsaw, the capital of Poland, is approximately 1